In [8]:
import numpy as np
import pandas as pd
data_target_cabang = {
    "kota": ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"],
    "target_bulanan": [45000000, 60000000, 55000000, 40000000, 30000000],
    "pic_cabang": ["Rani", "Joko", "Sari", "Bayu", "Fitri"],
}
np.random.seed(55)
n = 500
kategori_list = ["Elektronik", "Fashion", "Makanan & Minuman", "Kesehatan & Kecantikan", "Rumah Tangga"]
kota_list = ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"]
data_transaksi_t5 = {
    "order_id": [f"TRX-{i}" for i in range(n)],
    "kategori": np.random.choice(kategori_list, size=n),
    "kota": np.random.choice(kota_list, size=n),
    "unit_terjual": np.random.randint(1, 10, size=n),
    "harga_satuan": np.random.choice([25000, 50000, 75000, 100000, 150000], size=n),
}
pd.DataFrame(data_transaksi_t5).to_csv("transaksi_tugas5.csv", index=False)

!hdfs dfs -mkdir -p /user/revan18/tugas5
!hdfs dfs -put -f transaksi_tugas5.csv /user/revan18/tugas5/
print("Dataset siap. Tabel transaksi sudah diunggah ke HDFS: /user/revan18/tugas5/transaksi_tugas5.csv")
print("Simpan juga 'data_target_cabang' di atas — kalian akan membuatnya menjadi DataFrame sendiri di notebook tugas.")

Dataset siap. Tabel transaksi sudah diunggah ke HDFS: /user/revan18/tugas5/transaksi_tugas5.csv
Simpan juga 'data_target_cabang' di atas — kalian akan membuatnya menjadi DataFrame sendiri di notebook tugas.


In [12]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, count, row_number
from pyspark.sql.window import Window

spark = SparkSession.builder.appName("Tugas5").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

df_transaksi = (spark.read.option("header", True).option("inferSchema", True)
                .csv("hdfs://localhost:9000/user/revan18/tugas5/transaksi_tugas5.csv")
                .withColumn("pendapatan", col("unit_terjual") * col("harga_satuan")))
df_target = spark.createDataFrame(pd.DataFrame(data_target_cabang))
df_transaksi.show(5)
df_target.show()

+--------+--------------------+----------+------------+------------+----------+
|order_id|            kategori|      kota|unit_terjual|harga_satuan|pendapatan|
+--------+--------------------+----------+------------+------------+----------+
|   TRX-0|   Makanan & Minuman| Purworejo|           8|       75000|    600000|
|   TRX-1|          Elektronik|      Solo|           9|       75000|    675000|
|   TRX-2|Kesehatan & Kecan...|      Solo|           6|      100000|    600000|
|   TRX-3|             Fashion|Yogyakarta|           6|      100000|    600000|
|   TRX-4|          Elektronik|Yogyakarta|           2|       75000|    150000|
+--------+--------------------+----------+------------+------------+----------+
only showing top 5 rows

+----------+--------------+----------+
|      kota|target_bulanan|pic_cabang|
+----------+--------------+----------+
|  Magelang|      45000000|      Rani|
|Yogyakarta|      60000000|      Joko|
|  Semarang|      55000000|      Sari|
|      Solo|      400

In [13]:
#A.Join dan perbandingan target
ringkasan_kota = df_transaksi.groupBy("kota").agg(
    spark_sum("pendapatan").alias("total_pendapatan")
)
hasil_A = (ringkasan_kota.join(df_target, "kota", "inner")
    .withColumn("pencapaian_persen", col("total_pendapatan") / col("target_bulanan") * 100)
    .orderBy(col("pencapaian_persen").desc()))
hasil_A.show()

+----------+----------------+--------------+----------+------------------+
|      kota|total_pendapatan|target_bulanan|pic_cabang| pencapaian_persen|
+----------+----------------+--------------+----------+------------------+
| Purworejo|        45650000|      30000000|     Fitri|152.16666666666669|
|      Solo|        33475000|      40000000|      Bayu|           83.6875|
|Yogyakarta|        47275000|      60000000|      Joko| 78.79166666666667|
|  Magelang|        31650000|      45000000|      Rani| 70.33333333333334|
|  Semarang|        38175000|      55000000|      Sari|  69.4090909090909|
+----------+----------------+--------------+----------+------------------+



In [14]:
#B. Window function — kategori terlaris per kota
pendapatan_kategori = df_transaksi.groupBy("kota", "kategori").agg(
    spark_sum("pendapatan").alias("total_pendapatan_kategori")
)
w = Window.partitionBy("kota").orderBy(col("total_pendapatan_kategori").desc())
hasil_B = (pendapatan_kategori.withColumn("peringkat", row_number().over(w))
           .filter(col("peringkat") == 1).orderBy("kota"))
hasil_B.show()

+----------+--------------------+-------------------------+---------+
|      kota|            kategori|total_pendapatan_kategori|peringkat|
+----------+--------------------+-------------------------+---------+
|  Magelang|Kesehatan & Kecan...|                  7275000|        1|
| Purworejo|Kesehatan & Kecan...|                 10075000|        1|
|  Semarang|        Rumah Tangga|                 11125000|        1|
|      Solo|Kesehatan & Kecan...|                  8425000|        1|
|Yogyakarta|             Fashion|                 13325000|        1|
+----------+--------------------+-------------------------+---------+



In [15]:
#C. Spark SQL
df_transaksi.createOrReplaceTempView("transaksi_t5")
df_target.createOrReplaceTempView("target_cabang")
hasil_C = spark.sql("""
    SELECT t.kota, d.pic_cabang, COUNT(*) AS jumlah_transaksi
    FROM transaksi_t5 t
    JOIN target_cabang d ON t.kota = d.kota
    GROUP BY t.kota, d.pic_cabang
    ORDER BY jumlah_transaksi DESC
""")
hasil_C.show()

+----------+----------+----------------+
|      kota|pic_cabang|jumlah_transaksi|
+----------+----------+----------------+
| Purworejo|     Fitri|             116|
|Yogyakarta|      Joko|             110|
|      Solo|      Bayu|              95|
|  Semarang|      Sari|              93|
|  Magelang|      Rani|              86|
+----------+----------+----------------+



#D.Kesimpulan
menurut pengamatan saya kota purworejo memiliki pencapaian target paling tinggi, yaitu 152,17%, sedangkan Semarang paling rendah dan memerlukan perhatian manajemen dengan persentase 69,41%. Untuk kategori produk, Kesehatan & Kecantikan menjadi kategori dengan pendapatan tertinggi di Magelang, Purworejo, dan Solo. Semarang memiliki kategori Rumah Tangga sebagai yang tertinggi, sedangkan Yogyakarta adalah Fashion. Dari hasil tersebut, data dapat digunakan untuk melihat kondisi setiap cabang dan mengetahui kategori produk yang paling banyak menghasilkan pendapatan.